# 03a — Klasyczne ML: ekstrakcja cech

Buduje reprezentacje tekstu i zapisuje do `data/features/`. Kolejność: 03a → 03b → 03c.

## 1. Setup

In [ ]:
import os
import time
import json
import pickle
import warnings
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, hstack as sparse_hstack, issparse

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics import (
    f1_score, hamming_loss, jaccard_score, accuracy_score, precision_score, recall_score,
)
from sklearn.model_selection import KFold
import lightgbm as lgb

from thesis_lib import evaluate, cache_features   # wspólne helpery (jeden plik do przeczytania)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300

print(f"sklearn, lightgbm, gensim — wszystkie klasyczne, bez transformerów.")

In [2]:
# --- Constants ---
EMOTIONS = ["radość", "smutek", "zaufanie", "wstręt", "strach", "gniew", "przeczuwanie", "zdziwienie"]
RANDOM_STATE = 42

PROCESSED_DIR = Path("../data/processed")
CACHE_DIR     = Path("../data/features")
RESULTS_DIR   = Path("../data/results")
FIGURES_DIR   = Path("../figures")
for d in [CACHE_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Cache:   {CACHE_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

Cache:   ../data/features
Results: ../data/results
Figures: ../figures


In [3]:
# --- Load processed data ---
tw_train = pd.read_csv(PROCESSED_DIR / "twitteremo_train.csv").reset_index(drop=True)
tw_val   = pd.read_csv(PROCESSED_DIR / "twitteremo_val.csv").reset_index(drop=True)
tw_test  = pd.read_csv(PROCESSED_DIR / "twitteremo_test.csv").reset_index(drop=True)

go_train = pd.read_csv(PROCESSED_DIR / "go_emotions_train.csv").reset_index(drop=True)
go_val   = pd.read_csv(PROCESSED_DIR / "go_emotions_val.csv").reset_index(drop=True)
go_test  = pd.read_csv(PROCESSED_DIR / "go_emotions_test.csv").reset_index(drop=True)

for df in [tw_train, tw_val, tw_test, go_train, go_val, go_test]:
    df["clean_text"] = df["clean_text"].fillna("")

y_tw_train = tw_train[EMOTIONS].values
y_tw_val   = tw_val[EMOTIONS].values
y_tw_test  = tw_test[EMOTIONS].values
y_go_train = go_train[EMOTIONS].values
y_go_val   = go_val[EMOTIONS].values
y_go_test  = go_test[EMOTIONS].values

print(f"TwitterEmo:    train={len(tw_train):,}, val={len(tw_val):,}, test={len(tw_test):,}")
print(f"GoEmotions PL: train={len(go_train):,}, val={len(go_val):,}, test={len(go_test):,}")

TwitterEmo:    train=28,684, val=5,737, test=1,435
GoEmotions PL: train=34,276, val=6,855, test=1,714


### 1.1 Metryki

## 2. Ekstrakcja cech

### 2.1 TF-IDF word (1,2)-gram

In [6]:
def fit_tfidf_word(train_texts, val_texts, test_texts) -> dict:
    vec = TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
    )
    return {
        "train": vec.fit_transform(train_texts),
        "val":   vec.transform(val_texts),
        "test":  vec.transform(test_texts),
    }


tw_tfidf_word = cache_features(
    "tw_tfidf_word", fit_tfidf_word,
    tw_train["clean_text"], tw_val["clean_text"], tw_test["clean_text"],
)
print(f"TF-IDF word: {tw_tfidf_word['train'].shape}")

TF-IDF word: (28684, 18398)


### 2.2 TF-IDF char (3,5)-gram

Na tekście **surowym**, nie lemmatyzowanym — n-gramy znakowe wychwytują polską fleksję bez jawnej lemmatyzacji. To jedna z osi porównania: normalizacja morfologiczna implicytna vs explicytna.

In [7]:
def fit_tfidf_char(train_texts, val_texts, test_texts) -> dict:
    vec = TfidfVectorizer(
        max_features=50_000,
        ngram_range=(3, 5),
        analyzer="char_wb",
        min_df=3,
        sublinear_tf=True,
        lowercase=True,
    )
    return {
        "train": vec.fit_transform(train_texts),
        "val":   vec.transform(val_texts),
        "test":  vec.transform(test_texts),
    }


tw_tfidf_char = cache_features(
    "tw_tfidf_char", fit_tfidf_char,
    tw_train["tekst"].fillna(""), tw_val["tekst"].fillna(""), tw_test["tekst"].fillna(""),
)
print(f"TF-IDF char: {tw_tfidf_char['train'].shape}")

TF-IDF char: (28684, 50000)


### 2.3 fastText (trenowany na danych)

`sg=1`, `min_count=3`, `vector_size=200`. N-gramy znakowe 3–6 obsługują OOV. Embedding dokumentu = średnia wektorów tokenów.

In [8]:
from gensim.models import FastText


def train_fasttext(train_texts: pd.Series) -> Any:
    sentences = [t.split() for t in train_texts if t]
    print(f"  Training fastText on {len(sentences):,} sentences...")
    model = FastText(
        sentences=sentences,
        vector_size=200,
        window=5,
        min_count=3,
        workers=1,            # workers=1 + seed -> deterministic embeddings (reproducibility)
        seed=RANDOM_STATE,
        sg=1,
        epochs=10,
        min_n=3,
        max_n=6,
    )
    return model


def doc_embedding(tokens: list[str], model) -> np.ndarray:
    if not tokens:
        return np.zeros(model.vector_size, dtype=np.float32)
    vectors = [model.wv[t] for t in tokens]
    return np.mean(vectors, axis=0).astype(np.float32)


def fit_fasttext(train_texts, val_texts, test_texts) -> dict:
    model = train_fasttext(train_texts)
    def encode(texts):
        return np.vstack([doc_embedding(str(t).split(), model) for t in texts])
    return {
        "train": encode(train_texts),
        "val":   encode(val_texts),
        "test":  encode(test_texts),
        "model": model,
    }


tw_ft = cache_features(
    "tw_fasttext", fit_fasttext,
    tw_train["tekst"].fillna(""), tw_val["tekst"].fillna(""), tw_test["tekst"].fillna(""),
)
print(f"fastText (skip-gram, 200d): {tw_ft['train'].shape}")

fastText (skip-gram, 200d): (28684, 200)


### 2.4 LSA

`TruncatedSVD` na TF-IDF word → 300 wymiarów.

In [9]:
def fit_lsa(tfidf_feats: dict, n_components: int = 300) -> dict:
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    train_lsa = svd.fit_transform(tfidf_feats["train"])
    val_lsa   = svd.transform(tfidf_feats["val"])
    test_lsa  = svd.transform(tfidf_feats["test"])
    print(f"  LSA explained variance: {svd.explained_variance_ratio_.sum():.3f}")
    return {
        "train": train_lsa.astype(np.float32),
        "val":   val_lsa.astype(np.float32),
        "test":  test_lsa.astype(np.float32),
        "svd": svd,
    }


tw_lsa = cache_features("tw_lsa", fit_lsa, tw_tfidf_word, n_components=300)
print(f"LSA: {tw_lsa['train'].shape}")

LSA: (28684, 300)


### 2.5 LDA — rozkłady tematów

30 tematów na `CountVectorizer` (LDA wymaga liczników, nie TF-IDF).

In [10]:
def fit_lda(train_texts, val_texts, test_texts, n_topics: int = 30) -> dict:
    cv = CountVectorizer(max_features=10_000, min_df=5, max_df=0.95)
    X_train = cv.fit_transform(train_texts)
    X_val   = cv.transform(val_texts)
    X_test  = cv.transform(test_texts)

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method="online",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_iter=10,
    )
    train_topics = lda.fit_transform(X_train)
    val_topics   = lda.transform(X_val)
    test_topics  = lda.transform(X_test)

    # Top words per topic — used in thesis to name topics
    topic_words = []
    feature_names = cv.get_feature_names_out()
    for topic_idx, topic in enumerate(lda.components_):
        top = topic.argsort()[-10:][::-1]
        topic_words.append([feature_names[i] for i in top])

    return {
        "train": train_topics.astype(np.float32),
        "val":   val_topics.astype(np.float32),
        "test":  test_topics.astype(np.float32),
        "topic_words": topic_words,
        "lda": lda,
        "cv": cv,
    }


tw_lda = cache_features("tw_lda", fit_lda,
    tw_train["clean_text"], tw_val["clean_text"], tw_test["clean_text"],
    n_topics=30,
)
print(f"LDA: {tw_lda['train'].shape}")
print(f"\nPrzykładowe tematy (top słowa):")
for i in range(5):
    print(f"  Topic {i:2d}: {' | '.join(tw_lda['topic_words'][i][:6])}")

LDA: (28684, 30)

Przykładowe tematy (top słowa):
  Topic  0: dziękować | prezes | kaczyński | jutro | zapraszać | liczyć
  Topic  1: polak | powiedzieć | znać | niemiec | sytuacja | granica
  Topic  2: czekać | rynek | gotowy | wspaniały | luty | ojczyzna
  Topic  3: ukraina | prezydent | polska | pisać | warszawa | premier
  Topic  4: kolejny | projekt | możliwość | zakończyć | pół | zwycięstwo


### 2.6 Statystyki tekstu

11 cech: długość (znaki, słowa, śr. długość słowa), gęstość interpunkcji, frakcja wielkich liter, liczba URL/@/#/cyfr w tekście surowym, obecność emoji.

In [11]:
import re

EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
    "\u2600-\u26FF\u2700-\u27BF]+", flags=re.UNICODE,
)

STAT_FEATURE_NAMES = [
    "n_chars", "n_words", "avg_word_len",
    "frac_uppercase", "n_exclamations", "n_questions",
    "n_dots", "n_commas", "n_urls", "n_mentions", "has_emoji",
]


def text_statistics(texts: pd.Series) -> np.ndarray:
    feats = []
    for text in texts:
        s = str(text) if isinstance(text, str) else ""
        words = s.split()
        n_chars = len(s)
        n_words = len(words)
        avg_wl = np.mean([len(w) for w in words]) if words else 0.0
        n_upper = sum(1 for c in s if c.isupper())
        frac_upper = n_upper / max(n_chars, 1)
        feats.append([
            n_chars, n_words, avg_wl, frac_upper,
            s.count("!"), s.count("?"), s.count("."), s.count(","),
            len(re.findall(r"http\S+", s)),
            len(re.findall(r"@\w+", s)),
            int(bool(EMOJI_RE.search(s))),
        ])
    arr = np.array(feats, dtype=np.float32)
    return arr


def fit_text_stats(train_texts, val_texts, test_texts) -> dict:
    X_train = text_statistics(train_texts)
    X_val   = text_statistics(val_texts)
    X_test  = text_statistics(test_texts)
    # Standardize using train stats
    scaler = StandardScaler()
    return {
        "train": scaler.fit_transform(X_train).astype(np.float32),
        "val":   scaler.transform(X_val).astype(np.float32),
        "test":  scaler.transform(X_test).astype(np.float32),
        "scaler": scaler,
    }


tw_stats = cache_features("tw_stats", fit_text_stats,
    tw_train["tekst"].fillna(""), tw_val["tekst"].fillna(""), tw_test["tekst"].fillna(""),
)
print(f"Hand-crafted statistics: {tw_stats['train'].shape}")
print(f"Cechy: {STAT_FEATURE_NAMES}")

Hand-crafted statistics: (28684, 11)
Cechy: ['n_chars', 'n_words', 'avg_word_len', 'frac_uppercase', 'n_exclamations', 'n_questions', 'n_dots', 'n_commas', 'n_urls', 'n_mentions', 'has_emoji']


### 2.7 NRC EmoLex (PL) — cechy afektywne

Udział tokenów otagowanych każdą z 8 emocji + pokrycie leksykonu. Taksonomia NRC pokrywa się 1:1 z etykietami zadania.

In [12]:
# --- NRC EmoLex (PL) affective features ---
# Per-document lexical features: fraction of tokens tagged with each Plutchik emotion
# (+ positive/negative sentiment) in NRC EmoLex, plus overall lexicon coverage.
NRC_PATH = Path("../lexicons/Polish-NRC-EmoLex.txt")
NRC_CATEGORIES = ["anger", "anticipation", "disgust", "fear", "joy",
                  "sadness", "surprise", "trust", "positive", "negative"]
NRC_EN2PL = {"anger": "gniew", "anticipation": "przeczuwanie", "disgust": "wstręt",
             "fear": "strach", "joy": "radość", "sadness": "smutek",
             "surprise": "zdziwienie", "trust": "zaufanie"}


def _load_nrc(path: Path) -> tuple[dict, set]:
    df = pd.read_csv(path, sep="\t")
    cl = {c.lower(): c for c in df.columns}
    pl_col = next(c for lc, c in cl.items() if "polish" in lc)
    df[pl_col] = df[pl_col].astype(str).str.lower()
    cat_words = {cat: set(df.loc[df[cl[cat]] == 1, pl_col]) for cat in NRC_CATEGORIES}
    return cat_words, set(df[pl_col])


NRC_WORDS, NRC_VOCAB = _load_nrc(NRC_PATH)
NRC_FEATURE_NAMES = [NRC_EN2PL.get(c, c) for c in NRC_CATEGORIES] + ["coverage"]


def nrc_features(texts: pd.Series) -> np.ndarray:
    feats = []
    for text in texts:
        tokens = str(text).split() if isinstance(text, str) else []
        n = max(len(tokens), 1)
        row = [sum(t in NRC_WORDS[cat] for t in tokens) / n for cat in NRC_CATEGORIES]
        row.append(sum(t in NRC_VOCAB for t in tokens) / n)  # coverage
        feats.append(row)
    return np.array(feats, dtype=np.float32)


def fit_nrc(train_texts, val_texts, test_texts) -> dict:
    return {
        "train": nrc_features(train_texts),
        "val":   nrc_features(val_texts),
        "test":  nrc_features(test_texts),
    }


tw_nrc = cache_features("tw_nrc", fit_nrc,
    tw_train["clean_text"], tw_val["clean_text"], tw_test["clean_text"],
)
print(f"NRC EmoLex: {tw_nrc['train'].shape} ({NRC_FEATURE_NAMES})")

NRC EmoLex: (28684, 11) (['gniew', 'przeczuwanie', 'wstręt', 'strach', 'radość', 'smutek', 'zdziwienie', 'zaufanie', 'positive', 'negative', 'coverage'])


### 2.8 Złożenie: TF-IDF + LSA + LDA + NRC + statystyki

In [13]:
def combine_all(sparse_feats, *dense_feats) -> dict:
    """Combine sparse TF-IDF with multiple L2-normalized dense feature blocks."""
    out = {}
    for split in ["train", "val", "test"]:
        dense = np.hstack([normalize(d[split], norm="l2") for d in dense_feats])
        out[split] = sparse_hstack([sparse_feats[split], csr_matrix(dense)]).tocsr()
    return out


tw_combined = combine_all(tw_tfidf_word, tw_lsa, tw_lda, tw_nrc, tw_stats)
print(f"Combined (TF-IDF + LSA + LDA + NRC + stats): {tw_combined['train'].shape}")

Combined (TF-IDF + LSA + LDA + NRC + stats): (28684, 18750)


In [14]:
# --- TF-IDF word + char (3,5) concatenated: best sparse word + char-subword morphology ---
# Łączy lematyzowane słowne (1,2)-gramy (semantyka) z char (3,5)-gramami surowego tekstu
# (polska morfologia / OOV). Zwykle bije każdą z reprezentacji z osobna.
tw_tfidf_wordchar = {
    split: sparse_hstack([tw_tfidf_word[split], tw_tfidf_char[split]]).tocsr()
    for split in ["train", "val", "test"]
}
print(f"TF-IDF word+char: {tw_tfidf_wordchar['train'].shape}")

TF-IDF word+char: (28684, 68398)


### 2.9 Podsumowanie reprezentacji

In [15]:
TW_FEATURES = {
    "tfidf_word":      tw_tfidf_word,
    "tfidf_char":      tw_tfidf_char,
    "tfidf_wordchar":  tw_tfidf_wordchar,
    "fasttext":        tw_ft,
    "lsa":             tw_lsa,
    "lda":             tw_lda,
    "stats":           tw_stats,
    "nrc":             tw_nrc,
    "combined":        tw_combined,
}

summary_rows = []
for name, feats in TW_FEATURES.items():
    X = feats["train"]
    summary_rows.append({
        "representation": name,
        "n_features": X.shape[1],
        "type": "sparse" if issparse(X) else "dense",
        "memory_MB": round(
            (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes if issparse(X) else X.nbytes) / 1e6, 2
        ),
    })
rep_summary = pd.DataFrame(summary_rows)
display(rep_summary)
rep_summary.to_csv(RESULTS_DIR / "representations_summary.csv", index=False)

,representation,n_features,type,memory_MB
0,tfidf_word,18398,sparse,3.40
1,tfidf_char,50000,sparse,82.49
2,tfidf_wordchar,68398,sparse,85.78
3,fasttext,200,dense,22.95
4,lsa,300,dense,34.42
5,lda,30,dense,3.44
6,stats,11,dense,1.26
7,nrc,11,dense,1.26
8,combined,18750,sparse,121.18


In [16]:
# --- Handoff: persist assembled features for 03b / 03c ---
with open(CACHE_DIR / "TW_FEATURES.pkl", "wb") as f:
    pickle.dump(TW_FEATURES, f)
print(f"Saved {CACHE_DIR / 'TW_FEATURES.pkl'} ({len(TW_FEATURES)} representations)")

Saved ../data/features/TW_FEATURES.pkl (9 representations)
